# House Price Predictor: Model Training

This notebook generates a synthetic dataset, trains a Linear Regression model, and pushes the saved model artifact to the Hugging Face Model Hub.

In [ ]:
!pip install pandas numpy scikit-learn huggingface_hub

## 1. Generate Synthetic Data
We'll create 500 rows of synthetic house price data.

In [ ]:
import pandas as pd
import numpy as np
import os

def generate_synthetic_data(num_samples=500, random_seed=42):
    np.random.seed(random_seed)
    
    size_sqft = np.random.uniform(500, 4000, num_samples)
    bedrooms = np.random.randint(1, 7, num_samples)
    bathrooms = np.random.randint(1, 5, num_samples)
    age_years = np.random.randint(0, 41, num_samples)
    location_score = np.random.randint(1, 11, num_samples)
    
    base_price = 20.0
    true_price = (
        base_price +
        (size_sqft * 0.02) +
        (bedrooms * 3.0) +
        (bathrooms * 2.5) -
        (age_years * 0.8) +
        (location_score * 8.0)
    )
    
    noise = np.random.normal(0, 10, num_samples)
    price_lakhs = np.clip(true_price + noise, 10.0, None)
    
    df = pd.DataFrame({
        'size_sqft': np.round(size_sqft, 1),
        'bedrooms': bedrooms,
        'bathrooms': bathrooms,
        'age_years': age_years,
        'location_score': location_score,
        'price_lakhs': np.round(price_lakhs, 2)
    })
    return df

os.makedirs('data', exist_ok=True)
df = generate_synthetic_data()
df.to_csv('data/house_prices.csv', index=False)
print(f"Generated {len(df)} records and saved to data/house_prices.csv")
df.head()

## 2. Train the Linear Regression Model

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import pickle
import json

features = ['size_sqft', 'bedrooms', 'bathrooms', 'age_years', 'location_score']
target = 'price_lakhs'

X = df[features]
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("Model Training Complete!")
print(f"MAE:  {mean_absolute_error(y_test, y_pred):.2f} Lakhs")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):.2f} Lakhs")
print(f"R2:   {r2_score(y_test, y_pred):.4f}")

os.makedirs('model', exist_ok=True)
with open('model/model.pkl', 'wb') as f:
    pickle.dump(model, f)

with open('model/feature_names.json', 'w') as f:
    json.dump(features, f)
print("Model saved to model/model.pkl")

## 3. Push to Hugging Face
Ensure you have logged in using `huggingface-cli login` before running this cell.

In [ ]:
from huggingface_hub import HfApi

# To run this cell successfully in Colab, you need an active Hugging Face token.
# Uncomment the next two lines to login interactively:
# from huggingface_hub import notebook_login
# notebook_login()

api = HfApi()
username = api.whoami()['name']
repo_id = f"{username}/house-price-predictor"

print(f"Creating repository: {repo_id}")
api.create_repo(repo_id=repo_id, exist_ok=True, repo_type="model")

print("Uploading model files...")
api.upload_folder(
    folder_path="model",
    repo_id=repo_id,
    repo_type="model",
    commit_message="Add Linear Regression model and feature names from Colab"
)
print(f"Model successfully pushed to https://huggingface.co/{repo_id}")